# Fair Stroke Risk Demo with LightGBM
Simplified educational notebook with comments.

In [13]:
!pip install lightgbm

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from lightgbm import LGBMClassifier

In [15]:
np.random.seed(42)
n=1000
df=pd.DataFrame({
    "age":np.random.randint(40,90,n),
    "diabetes":np.random.randint(0,2,n),
    "hypertension":np.random.randint(0,2,n),
    "bmi":np.random.normal(28,5,n),
    "race":np.random.choice(["White","Black"],size=n,p=[0.8,0.2]),
})
# Toy label (not medically valid)
logit=(df.age>70).astype(int)+df.diabetes+df.hypertension+(df.race=="Black").astype(int)
prob=1/(1+np.exp(-(logit-2)))
df["stroke"]=np.random.binomial(1,prob)

In [16]:
# STEP 2: One-hot encode race
X=pd.get_dummies(df.drop(columns="stroke"),drop_first=True)
y=df["stroke"]

X_train,X_tune,y_train,y_tune=train_test_split(
    X,y,test_size=0.3,random_state=42
)

In [17]:
# STEP 3: Train LightGBM
model=LGBMClassifier(n_estimators=100,random_state=42)
model.fit(X_train,y_train)

[LightGBM] [Info] Number of positive: 279, number of negative: 421
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000239 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 291
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.398571 -> initscore=-0.411421
[LightGBM] [Info] Start training from score -0.411421
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

LGBMClassifier(random_state=42)

In [18]:
# STEP 4: Evaluate AUROC
probs=model.predict_proba(X_tune)[:,1]
overall_auc=roc_auc_score(y_tune,probs)
print("Overall AUROC:",overall_auc)

Overall AUROC: 0.6385626073633873


In [19]:
# STEP 5: Fairness-aware score
# Compute AUROC separately for White and Black groups.
race_black=X_tune["race_White"]==0

auc_black=roc_auc_score(y_tune[race_black],probs[race_black])
auc_white=roc_auc_score(y_tune[~race_black],probs[~race_black])

disparity=abs(auc_white-auc_black)
lam=2.0

fair_score=overall_auc-lam*disparity

print("White AUROC:",auc_white)
print("Black AUROC:",auc_black)
print("Disparity:",disparity)
print("Fairness-aware tuning score:",fair_score)

White AUROC: 0.6313887454827053
Black AUROC: 0.65625
Disparity: 0.024861254517294706
Fairness-aware tuning score: 0.5888400983287979


In [20]:
# STEP 6: Race-specific thresholds
# Youden's J = sensitivity + specificity - 1
def best_threshold(y_true,p):
    fpr,tpr,thr=roc_curve(y_true,p)
    j=tpr-fpr
    idx=np.argmax(j)
    return thr[idx],j[idx]

thr_black,_=best_threshold(y_tune[race_black],probs[race_black])
thr_white,_=best_threshold(y_tune[~race_black],probs[~race_black])

print("Threshold (Black):",thr_black)
print("Threshold (White):",thr_white)

Threshold (Black): 0.5327833906484342
Threshold (White): 0.42810705830894685


In [21]:
# STEP 7: Apply race-specific thresholds
pred=np.zeros(len(probs),dtype=int)
pred[race_black]=probs[race_black]>=thr_black
pred[~race_black]=probs[~race_black]>=thr_white

print("Finished demo.")

Finished demo.
